In [50]:
import pandas as pd
import re
import unicodedata
from collections import defaultdict
from tqdm import tqdm

tqdm.pandas()
df = pd.read_csv(r"C:\Users\glgas\OneDrive - UCSF\Desktop\MATH 4100\ANAC_loc.csv")
routes = pd.read_csv(r"C:\Users\glgas\OneDrive - UCSF\Desktop\MATH 4100\route mapping\routes_matcher_combined.csv")


def norm(s):
    if s is None or pd.isna(s):
        return ""
    s = str(s)
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = s.lower()
    s = re.sub(r"\s+", " ", s).strip()
    return s

def canon_country(x):
    x = norm(x)
    if x in {"ca", "canada"}:
        return "Canada"
    if x in {"us", "usa", "united states", "united states of america"}:
        return "United States"
    if x in {"mx", "mexico"}:
        return "Mexico"
    return str(x).strip()

# Province / region / state name cleanup so the two CSVs line up better
PROVINCE_MAP = {
    "yukon": "Yukon Territory",
    "yukon territory": "Yukon Territory",
    "northwest territories": "Northwest Territories",
    "newfoundland": "Newfoundland and Labrador",
    "labrador": "Newfoundland and Labrador",
    "quebec": "Quebec",
    "british columbia": "British Columbia",
    "alberta": "Alberta",
    "ontario": "Ontario",
    "nova scotia": "Nova Scotia",
    "new brunswick": "New Brunswick",
    "manitoba": "Manitoba",
    "saskatchewan": "Saskatchewan",
    "nunavut": "Nunavut",
    "prince edward island": "Prince Edward Island",
    "baffin island": "Baffin Island",
    "new mexico": "New Mexico",
    "california": "California",
    "colorado": "Colorado",
    "washington": "Washington",
}

PROVINCE_MAP.update({
    "aguascalientes": "Aguascalientes",
    "baja california": "Baja California",
    "baja california sur": "Baja California Sur",
    "campeche": "Campeche",
    "chiapas": "Chiapas",
    "chihuahua": "Chihuahua",
    "coahuila": "Coahuila",
    "colima": "Colima",
    "durango": "Durango",
    "guanajuato": "Guanajuato",
    "guerrero": "Guerrero",
    "hidalgo": "Hidalgo",
    "jalisco": "Jalisco",
    "mexico city": "Mexico City",
    "méxico city": "Mexico City",
    "mexico state": "México",
    "méxico": "México",
    "michoacan": "Michoacán",
    "michoacán": "Michoacán",
    "morelos": "Morelos",
    "nayarit": "Nayarit",
    "nuevo leon": "Nuevo León",
    "nuevo león": "Nuevo León",
    "oaxaca": "Oaxaca",
    "puebla": "Puebla",
    "queretaro": "Querétaro",
    "querétaro": "Querétaro",
    "quintana roo": "Quintana Roo",
    "san luis potosi": "San Luis Potosí",
    "san luis potosí": "San Luis Potosí",
    "sinaloa": "Sinaloa",
    "sonora": "Sonora",
    "tabasco": "Tabasco",
    "tamaulipas": "Tamaulipas",
    "tlaxcala": "Tlaxcala",
    "veracruz": "Veracruz",
    "yucatan": "Yucatán",
    "yucatán": "Yucatán",
    "zacatecas": "Zacatecas",
})

PROVINCE_MAP.update({
    # --- US States ---
    "al": "Alabama", "alabama": "Alabama",
    "ak": "Alaska", "alaska": "Alaska",
    "az": "Arizona", "arizona": "Arizona",
    "ar": "Arkansas", "arkansas": "Arkansas",
    "ca": "California", "california": "California",
    "co": "Colorado", "colorado": "Colorado",
    "ct": "Connecticut", "connecticut": "Connecticut",
    "de": "Delaware", "delaware": "Delaware",
    "fl": "Florida", "florida": "Florida",
    "ga": "Georgia", "georgia": "Georgia",
    "hi": "Hawaii", "hawaii": "Hawaii",
    "id": "Idaho", "idaho": "Idaho",
    "il": "Illinois", "illinois": "Illinois",
    "in": "Indiana", "indiana": "Indiana",
    "ia": "Iowa", "iowa": "Iowa",
    "ks": "Kansas", "kansas": "Kansas",
    "ky": "Kentucky", "kentucky": "Kentucky",
    "la": "Louisiana", "louisiana": "Louisiana",
    "me": "Maine", "maine": "Maine",
    "md": "Maryland", "maryland": "Maryland",
    "ma": "Massachusetts", "massachusetts": "Massachusetts",
    "mi": "Michigan", "michigan": "Michigan",
    "mn": "Minnesota", "minnesota": "Minnesota",
    "ms": "Mississippi", "mississippi": "Mississippi",
    "mo": "Missouri", "missouri": "Missouri",
    "mt": "Montana", "montana": "Montana",
    "ne": "Nebraska", "nebraska": "Nebraska",
    "nv": "Nevada", "nevada": "Nevada",
    "nh": "New Hampshire", "new hampshire": "New Hampshire",
    "nj": "New Jersey", "new jersey": "New Jersey",
    "nm": "New Mexico", "new mexico": "New Mexico",
    "ny": "New York", "new york": "New York",
    "nc": "North Carolina", "north carolina": "North Carolina",
    "nd": "North Dakota", "north dakota": "North Dakota",
    "oh": "Ohio", "ohio": "Ohio",
    "ok": "Oklahoma", "oklahoma": "Oklahoma",
    "or": "Oregon", "oregon": "Oregon",
    "pa": "Pennsylvania", "pennsylvania": "Pennsylvania",
    "ri": "Rhode Island", "rhode island": "Rhode Island",
    "sc": "South Carolina", "south carolina": "South Carolina",
    "sd": "South Dakota", "south dakota": "South Dakota",
    "tn": "Tennessee", "tennessee": "Tennessee",
    "tx": "Texas", "texas": "Texas",
    "ut": "Utah", "utah": "Utah",
    "vt": "Vermont", "vermont": "Vermont",
    "va": "Virginia", "virginia": "Virginia",
    "wa": "Washington", "washington": "Washington",
    "wv": "West Virginia", "west virginia": "West Virginia",
    "wi": "Wisconsin", "wisconsin": "Wisconsin",
    "wy": "Wyoming", "wyoming": "Wyoming",
})

def canon_province(x):
    x_norm = norm(x)
    return PROVINCE_MAP.get(x_norm, str(x).strip())


# Generic alias blacklist
COUNTRY_NAMES = {
    "canada",
    "united states",
    "united states of america",
    "usa",
    "us",
    "mexico",
    "mx",
}

CANADA_PROVINCES = {
    "alberta",
    "british columbia",
    "manitoba",
    "new brunswick",
    "newfoundland and labrador",
    "newfoundland",
    "labrador",
    "nova scotia",
    "ontario",
    "prince edward island",
    "quebec",
    "saskatchewan",
    "northwest territories",
    "nunavut",
    "yukon",
    "yukon territory",
}

US_STATES = {
    "alabama", "alaska", "arizona", "arkansas", "california", "colorado",
    "connecticut", "delaware", "district of columbia", "florida", "georgia",
    "hawaii", "idaho", "illinois", "indiana", "iowa", "kansas", "kentucky",
    "louisiana", "maine", "maryland", "massachusetts", "michigan",
    "minnesota", "mississippi", "missouri", "montana", "nebraska", "nevada",
    "new hampshire", "new jersey", "new mexico", "new york", "north carolina",
    "north dakota", "ohio", "oklahoma", "oregon", "pennsylvania",
    "rhode island", "south carolina", "south dakota", "tennessee", "texas",
    "utah", "vermont", "virginia", "washington", "west virginia", "wisconsin",
    "wyoming"
}

MEXICO_STATES = {
    "aguascalientes", "baja california", "baja california sur", "campeche",
    "chiapas", "chihuahua", "coahuila", "colima", "durango", "guanajuato",
    "guerrero", "hidalgo", "jalisco", "mexico city", "méxico city", "michoacan",
    "michoacán", "morelos", "nayarit", "nuevo leon", "nuevo león", "oaxaca",
    "puebla", "queretaro", "querétaro", "quintana roo", "san luis potosi",
    "san luis potosí", "sinaloa", "sonora", "tabasco", "tamaulipas",
    "tlaxcala", "veracruz", "yucatan", "yucatán", "zacatecas", "méxico", "mexico"
}

GENERIC_HIERARCHY_TERMS = {
    "north america",
    "international",
    "area",
    "region",
    "province",
    "state",
    "park",
    "national park",
    "wilderness",
    "forest",
}

GENERIC_ALIAS_TERMS = {
    *COUNTRY_NAMES,
    *CANADA_PROVINCES,
    *US_STATES,
    *MEXICO_STATES,
    *GENERIC_HIERARCHY_TERMS,
}

def is_generic_alias(term):
    return norm(term) in GENERIC_ALIAS_TERMS

# Standardize both dataframes

df["Country_std"] = df["Country"].apply(canon_country)
df["State_Province_std"] = df["State_Province"].apply(canon_province)

routes["country_std"] = routes["country"].apply(canon_country)
routes["province_std"] = routes["province"].apply(canon_province)

# Route aliases into lists
def split_aliases(x):
    if pd.isna(x) or str(x).strip() == "":
        return []
    return [a.strip() for a in str(x).split("|") if a.strip()]

routes["alias_list"] = routes["aliases"].apply(split_aliases)


# Build province-specific lookup
# key = (country_std, province_std)
# value = { normalized term: info }


def build_region_lookup(routes_df):
    region_lookup = defaultdict(list)

    for _, row in routes_df.iterrows():
        key = (row["country_std"], row["province_std"])

        info = {
            "matched_name": row["name"],
            "country": row["country_std"],
            "province": row["province_std"],
            "lat": row["lat"],
            "lon": row["lon"],
            "route": row.get("Route", None),
            "url": row.get("URL", None),
        }

        ordered_terms = [row["name"]] + list(row["alias_list"])

        for term in ordered_terms:
            term_norm = norm(term)
            if not term_norm:
                continue

            # keep the canonical name even if it looks generic,
            # but filter generic aliases
            if term_norm != norm(row["name"]) and is_generic_alias(term_norm):
                continue

            region_lookup[key].append({
                "term": term_norm,
                "info": info
            })

    return region_lookup

region_lookup = build_region_lookup(routes)


# Exact matcher for one row

def extract_location_for_row(text, country, province, region_lookup):
    if not isinstance(text, str) or not text.strip():
        return None

    key = (canon_country(country), canon_province(province))
    entries = region_lookup.get(key)

    if not entries:
        return None

    text_norm = norm(text)

    for entry in entries:
        term = entry["term"]
        pattern = r"(?<!\w)" + re.escape(term) + r"(?!\w)"
        if re.search(pattern, text_norm):
            result = entry["info"].copy()
            result["matched_phrase"] = term
            result["score"] = 100
            return result

    return None

def extract_location_fields(row):
    result = extract_location_for_row(
        row["Accident Title"],
        row["Country"],
        row["State_Province"],
        region_lookup
    )

    if result:
        matched = routes[
            (routes["name"] == result["matched_name"]) &
            (routes["country_std"] == canon_country(row["Country"])) &
            (routes["province_std"] == canon_province(row["State_Province"]))
        ].head(1)

        aliases = ""
        if len(matched) > 0:
            aliases = matched.iloc[0]["aliases"]
            if pd.isna(aliases):
                aliases = ""

        display_location = result["matched_name"]
        if aliases:
            display_location = f"{result['matched_name']} | aliases: {aliases}"

        return display_location, result["matched_phrase"], result["lat"], result["lon"]

    return None, None, None, None


df[["location_name", "matched_phrase", "lat", "lon"]] = df.progress_apply(
    extract_location_fields,
    axis=1,
    result_type="expand"
)




100%|██████████| 30/30 [00:00<00:00, 141.44it/s]


   Country    State_Province  \
0       CA  British Columbia   
1       CA           Alberta   
2       CA  British Columbia   
3       CA           Alberta   
4       CA           Alberta   
5       CA            Quebec   
6       CA  British Columbia   
7       CA            Quebec   
8       CA           Alberta   
9       CA           Ontario   
10      CA            Quebec   
11      CA           Alberta   
12      CA           Alberta   
13      CA           Alberta   
14      CA  British Columbia   
15      CA           Alberta   
16      CA           Alberta   
17      CA           Alberta   
18      CA  British Columbia   
19      CA           Alberta   
20      CA           Alberta   
21      CA           Alberta   
22      CA           Alberta   
23      CA           Alberta   
24      CA           Alberta   
25      CA           Alberta   
26      CA           Alberta   
27      CA           Alberta   
28      CA           Alberta   
29      CA           Alberta   

       

100%|██████████| 2770/2770 [00:18<00:00, 148.73it/s]

  ID                                     Accident Title  Publication Year  \
0  2  Failure of Rappel—Failure to Check System, Bri...            1990.0   
1  3  Fall into Crevasse, Climbing Alone, Inadequate...            1990.0   
2  4  Fall into Crevasse, Climbing Unroped, British ...            1990.0   
3  5  Fall Into Crevasse, Unroped, Inadequate Equipm...            1990.0   
4  6  Fall into Moat, Descending Unroped, Poor Posit...            1990.0   

                                                Text  \
0  British Columbia, Squamish, Smoke Bluffs\nOn M...   
1  Alberta, Rocky Mountains, Crowfoot Mountain\nO...   
2  British Columbia, Bugaboo Mountains, Bugaboo S...   
3  On the afternoon of March 29, 1989, four ski t...   
4  Alberta, Rocky Mountains, Mount Nublock\nLate ...   

                                        Tags Applied Country  \
0  Experienced, Serious, Descent, Roped, Top-Rope...      CA   
1  Experienced, Minor, Unroped , Solo, Climbing A...      CA   
2  Minor

In [54]:
df.to_csv(r"C:\Users\glgas\OneDrive - UCSF\Desktop\MATH 4100\ANAC_mp_test.csv", index=False)
#df.isna().mean() * 100